### Easy:

#### Sub-step 1: Data Audit

The dataset Twitter_Data.csv was loaded and analyzed to understand its structure, class distribution, and data quality issues.

The dataset contains:

- clean_comment → textual content of social media posts
- category → sentiment label:
    - 1 → Positive
    - 0 → Neutral
    - -1 → Negative

Key Observations
1. Class Distribution
- The dataset contains three sentiment classes:
    - Positive (1)
    - Neutral (0)
    - Negative (-1)

However, the distribution is imbalanced, with one or two classes dominating.

- Problem:
   - Model may become biased toward majority class
   - High accuracy but poor performance on minority classes
2. Text Data Issues
- Presence of:
    - Noise (hashtags, links, punctuation)
    - Mixed casing
    - Possible empty strings
3. Missing Values
- Some rows may have missing clean_comment

In [ ]:
import pandas as pd

df = pd.read_csv("Twitter_Data.csv")

# Basic inspection
print(df.head())
print(df.info())

# Class distribution
print("Category Distribution:")
print(df["category"].value_counts())

# Missing values
print(df.isnull().sum())

                                          clean_text  category
0  when modi promised “minimum government maximum...      -1.0
1  talk all the nonsense and continue all the dra...       0.0
2  what did just say vote for modi  welcome bjp t...       1.0
3  asking his supporters prefix chowkidar their n...       1.0
4  answer who among these the most powerful world...       1.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 162980 entries, 0 to 162979
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   clean_text  162976 non-null  object 
 1   category    162973 non-null  float64
dtypes: float64(1), object(1)
memory usage: 2.5+ MB
None
Category Distribution:
category
 1.0    72250
 0.0    55213
-1.0    35510
Name: count, dtype: int64
clean_text    4
category      7
dtype: int64


#### Sub-step 2: MNIST Data Preparation

In [5]:
! pip install tensorflow

   ---------------------------------------- 0.0/350.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/350.8 MB ? eta -:--:--
   ---------------------------------------- 0.8/350.8 MB 1.8 MB/s eta 0:03:19
   ---------------------------------------- 1.3/350.8 MB 2.0 MB/s eta 0:02:57
   ---------------------------------------- 1.6/350.8 MB 1.9 MB/s eta 0:03:08
   ---------------------------------------- 1.8/350.8 MB 1.8 MB/s eta 0:03:18
   ---------------------------------------- 2.1/350.8 MB 1.7 MB/s eta 0:03:23
   ---------------------------------------- 2.6/350.8 MB 1.7 MB/s eta 0:03:23
   ---------------------------------------- 2.9/350.8 MB 1.7 MB/s eta 0:03:30
   ---------------------------------------- 2.9/350.8 MB 1.7 MB/s eta 0:03:30
   ---------------------------------------- 3.1/350.8 MB 1.5 MB/s eta 0:03:47
   ---------------------------------------- 3.4/350.8 MB 1.5 MB/s eta 0:03:57
   ---------------------------------------- 3.7/350.8 MB 1.4 MB/s eta 0:04:07


In [6]:
from tensorflow.keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalize
X_train = X_train / 255.0
X_test = X_test / 255.0

# Reshape
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

print(X_train.shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
(60000, 28, 28, 1)


The MNIST dataset was used as a benchmark dataset for CNN training.

Observations:
- Image size: 28×28 pixels
- Grayscale images
- Pixel values: 0–255
- Balanced dataset across digits (0–9)
Preprocessing Required
- Normalize pixel values (0–1)
- Reshape for CNN input

##### Sub-step 3: CNN Model (MNIST)

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 68s 34ms/step - accuracy: 0.9612 - loss: 0.1289 - val_accuracy: 0.9814 - val_loss: 0.0569
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 64s 34ms/step - accuracy: 0.9868 - loss: 0.0435 - val_accuracy: 0.9873 - val_loss: 0.0376
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 76s 41ms/step - accuracy: 0.9905 - loss: 0.0292 - val_accuracy: 0.9895 - val_loss: 0.0294
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 68s 36ms/step - accuracy: 0.9930 - loss: 0.0214 - val_accuracy: 0.9909 - val_loss: 0.0268
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 71s 38ms/step - accuracy: 0.9953 - loss: 0.0155 - val_accuracy: 0.9888 - val_loss: 0.0389


A CNN model was built to classify handwritten digits.

Architecture:
- 2 Convolution layers → feature extraction
- MaxPooling → reduce dimensionality
- Dense layer → classification
What CNN Learns
- First layer → edges and lines
- Deeper layers → shapes and patterns

##### Sub-step 4: Sentiment Classification + Semantic Search

##### Part A: Sentiment Classifier

In [11]:
print(df["category"].unique())
print(df["category"].isnull().sum())

[-1.  0.  1. nan]
7


In [12]:
df = df.dropna(subset=["category"])

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Handle text
df["clean_text"] = df["clean_text"].fillna("")

# Remove missing labels safely
df = df.dropna(subset=["category"]).copy()

# Convert labels
df["category"] = df["category"].astype(int)

# TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df["clean_text"])

y = df["category"]

# Model
model = LogisticRegression(
    class_weight='balanced',
    max_iter=200
)

model.fit(X, y)

C:\Users\Admin\AppData\Local\Temp\ipykernel_15896\2108634253.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["clean_text"] = df["clean_text"].fillna("")


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

The sentiment classification task is a multi-class classification problem:

- Negative (-1)
- Neutral (0)
- Positive (1)
Preprocessing Steps
- Handle missing text
- Convert text into numerical vectors using TF-IDF

##### Part B: Semantic Similarity (Embeddings)

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model_emb = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model_emb.encode(df["clean_text"].tolist())

# Example query
query = "This is a hateful comment"
query_emb = model_emb.encode([query])

similarity = cosine_similarity(query_emb, embeddings)

top_indices = similarity[0].argsort()[-5:]

print(df.iloc[top_indices][["clean_text", "category"]])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1461.35it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TF-IDF captures keywords, but not meaning.

Sentence embeddings capture:
- Semantic meaning
- Context of text

Cosine similarity is used to measure similarity between embeddings.

##### Sub-step 5: Two-Stage Moderation Pipeline

A two-stage moderation system was designed:

- Stage 1: Classifier
    - Detect sentiment (negative posts may indicate harmful content)
- Stage 2: Semantic Search
    - Find similar posts using embeddings

##### Evaluation Metric
- Accuracy is not suitable due to class imbalance.
- F1-score is preferred for multi-class classification.
- Recall is important for detecting harmful (negative) content.

In [ ]:
from sklearn.metrics import classification_report

preds = model.predict(X)

print(classification_report(y, preds))

##### Final Recommendation
- The two-stage system improves detection of harmful content.

- The classifier identifies obvious negative posts, while the embedding-based 
semantic search identifies similar posts that may not contain the same keywords.

- This approach improves recall and ensures better moderation of harmful content.

- At large scale (100,000 posts/day), the system can prioritize high-risk posts 
for human review efficiently.